In [ ]:

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder


In [ ]:


# 文件路径
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

# 读取数据
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 查看数据的基本情况
train_df.head(), test_df.head()



(      id  no_of_adults  ...  no_of_special_requests  booking_status
 0  15559             2  ...                       2               0
 1  32783             2  ...                       1               0
 2  11797             3  ...                       0               1
 3  39750             2  ...                       1               1
 4  28711             2  ...                       0               1
 
 [5 rows x 19 columns],
       id  no_of_adults  ...  no_of_special_requests  booking_status
 0   8768             2  ...                       1               0
 1  38340             2  ...                       0               1
 2   7104             2  ...                       0               0
 3  36898             2  ...                       3               0
 4   9747             2  ...                       1               0
 
 [5 rows x 19 columns])

In [ ]:


# 检查数据类型和缺失值
train_df.info(), test_df.info()

# 描述性统计
train_df.describe(include='all'), test_df.describe(include='all')




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

                            8420 non-null   int64  
 1   no_of_adults                          8420 non-null   int64  
 2   no_of_children                        8420 non-null   int64  
 3   no_of_weekend_nights                  8420 non-null   int64  
 4   no_of_week_nights                     8420 non-null   int64  
 5   type_of_meal_plan                     8420 non-null   int64  
 6   required_car_parking_space            8420 non-null   int64  
 7   room_type_reserved                    8420 non-null   int64  
 8   lead_time                             8420 non-null   int64  
 9   arrival_year                          8420 non-null   int64  
 10  arrival_month                         8420 non-null   int64  
 11  arrival_date                          8420 non-null   int64  
 12  market_segment_type                   84

In [ ]:



# 移除 'id' 列
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])

# 划分特征和目标变量
X_train = train_df.drop(columns=['booking_status'])
y_train = train_df['booking_status']

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# 检查预处理后的数据
X_train.head(), X_val.head(), y_train.head(), y_val.head()



(       no_of_adults  no_of_children  ...  avg_price_per_room  no_of_special_requests
 28585             1               0  ...               78.30                       2
 11854             2               0  ...              101.33                       0
 25872             2               0  ...              102.30                       2
 31757             2               0  ...               88.60                       1
 26401             2               0  ...               72.25                       0
 
 [5 rows x 17 columns],
        no_of_adults  no_of_children  ...  avg_price_per_room  no_of_special_requests
 19413             2               0  ...               95.69                       2
 18726             2               0  ...               90.95                       0
 23038             2               0  ...               99.45                       1
 20686             2               0  ...               97.33                       3
 11300             2        

In [ ]:


# 创建LightGBM数据集
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# 设置模型参数
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# 训练模型
gbm = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=100, early_stopping_rounds=10)

# 评估模型
y_val_pred = gbm.predict(X_val, num_iteration=gbm.best_iteration)
roc_auc_val = roc_auc_score(y_val, y_val_pred)
roc_auc_val




---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[6], line 18
     16 pass
     17 print('##active_line19##')
---> 18 gbm = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=100, early_stopping_rounds=10)
     19 print('##active_line20##')
     20 pass

TypeError: train() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:



# 训练模型
gbm = lgb.train(params, train_data, num_boost_round=100)

# 评估模型
y_val_pred = gbm.predict(X_val, num_iteration=gbm.best_iteration)
roc_auc_val = roc_auc_score(y_val, y_val_pred)
roc_auc_val




np.float64(0.8859465452148378)

In [ ]:



# 移除测试集中的 'id' 列（如果存在）
if 'id' in test_df.columns:
    test_df = test_df.drop(columns=['id'])

# 使用训练好的模型对测试集进行预测
y_test_pred = gbm.predict(test_df, num_iteration=gbm.best_iteration)

# 将预测结果保存到文件
test_df['booking_status_pred'] = y_test_pred
test_df[['booking_status_pred']].to_csv('reservation_cancellation_predictions.csv', index=False)

# 检查前几行预测结果
test_df[['booking_status_pred']].head()




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

  1181         num_iteration=num_iteration,
   1182         predict_type=predict_type,
   1183     )
   1184 elif isinstance(data, np.ndarray):
-> 1185     preds, nrow = self.__pred_for_np2d(
   1186         mat=data,
   1187         start_iteration=start_iteration,
   1188         num_iteration=num_iteration,
   1189         predict_type=predict_type,
   1190     )
   1191 elif _is_pyarrow_table(data):
   1192     preds, nrow = self.__pred_for_pyarrow_table(
   1193         table=data,
   1194         start_iteration=start_iteration,
   1195         num_iteration=num_iteration,
   1196         predict_type=predict_type,
   1197     )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\lightgbm\basic.py:1344, in _InnerPredictor.__pred_for_np2d(self, mat, start_iteration, num_iteration, 

In [ ]:



# 确保测试集和训练集的特征数量一致
test_df = test_df.drop(columns=['id'])

# 使用训练好的模型对测试集进行预测
y_test_pred = gbm.predict(test_df, num_iteration=gbm.best_iteration)

# 将预测结果保存到文件
test_df['booking_status_pred'] = y_test_pred
test_df[['booking_status_pred']].to_csv('reservation_cancellation_predictions.csv', index=False)

# 检查前几行预测结果
test_df[['booking_status_pred']].head()




---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
Cell In[9], line 4
      2 pass
      3 print('##active_line2##')
----> 4 test_df = test_df.drop(columns=['id'])
      5 print('##active_line3##')
      6 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\frame.py:5581, in DataFrame.drop(self, labels, axis, index, columns, level, inplace, errors)
   5433 def drop(
   5434     self,
   5435     labels: IndexLabel | None = None,
   (...)
   5442     errors: IgnoreRaise = "raise",
   5443 ) -> DataFrame | None:
   5444     """
   5445     Drop specified labels from rows or columns.
   5446 
   (...)
   5579             weight  1.0     0.8
   5580     """
-> 5581     return super().drop(
   5582         labels=labels,
   5583         axis=axis,
   5584         index=index,
   5585         columns=columns,
   5586         level=level,
 

In [ ]:



# 检查测试集的列名
test_df.columns




Index(['no_of_adults', 'no_of_children', 'no_of_weekend_nights',
       'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space',
       'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month',
       'arrival_date', 'market_segment_type', 'repeated_guest',
       'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled',
       'avg_price_per_room', 'no_of_special_requests', 'booking_status'],
      dtype='object')

In [ ]:




# 使用训练好的模型对测试集进行预测
y_test_pred = gbm.predict(test_df.drop(columns=['booking_status']), num_iteration=gbm.best_iteration)

# 将预测结果保存到文件
test_df['booking_status_pred'] = y_test_pred
test_df[['booking_status_pred']].to_csv('reservation_cancellation_predictions.csv', index=False)

# 检查前几行预测结果
test_df[['booking_status_pred']].head()





Displayed on the user's machine.